##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Check citation faithfulness in RAG

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Citation_Faithfulness_Check.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

When a Gemini RAG answer cites its sources — via grounding metadata, or by asking
for **structured citations** with a response schema — the citations can fail in ways
that read as completely authoritative, and an LLM asked "does this quote support the
claim?" waves them through:

- **fabricated** — a quoted span that appears in no source document;
- **frankenquote** — every word is real, but the exact span was never written
  contiguously in the source;
- **misattributed** — a real span, but attributed to the wrong document.

This notebook shows a *cheap deterministic detector → expensive judge* pattern:

1. Ask Gemini, via **structured output** (a `response_schema`), to return each claim
   with the `document_id` it relies on and a short **verbatim quote**.
2. Run a **0-token verbatim gate** (pure Python — no model, no API key) that checks
   each quote appears in the cited document. This rejects the three failure modes
   above and runs offline.
3. Only for quotes that pass the gate, optionally call a **burden-of-proof judge** —
   fabrications never reach it, so they cost zero tokens.

The gate is inlined here; its standalone, framework-agnostic version (gate +
burden-of-proof judge) lives at
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate).

## The verbatim gate (deterministic, runs offline)

In [ ]:
import re


def normalize(text: str) -> str:
    """Case/typography/whitespace-insensitive form for verbatim matching."""
    text = text.lower()
    text = re.sub(r"[‘’]", "'", text)
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[–—]", "-", text)
    text = re.sub(r"[^a-z0-9%.]+", " ", text)
    return " ".join(text.split())


def gate(quote: str, cited_doc_id: str, docs: dict) -> str:
    """Return 'found' | 'misattributed' | 'not_found'. Fails closed on empty quotes."""
    q = normalize(quote)
    if not q:
        return "not_found"
    cited = docs.get(cited_doc_id)
    if cited is not None and q in normalize(cited):
        return "found"
    if any(q in normalize(t) for d, t in docs.items() if d != cited_doc_id):
        return "misattributed"
    return "not_found"

### Verify it offline on structured citations

No API key needed here — this is the shape of citations a `response_schema` returns,
with one faithful citation and the three planted failure modes.

In [ ]:
DOCS = {
    "doc_0": "Gemini 2.5 Pro has a context window of up to 1 million tokens.",
    "doc_1": "Gemini 2.5 Flash is optimized for low latency and high throughput.",
}

CITATIONS = [
    {"claim": "Gemini 2.5 Pro supports a 1M token context.", "document_id": "doc_0",
     "quote": "context window of up to 1 million tokens"},                     # faithful
    {"claim": "Flash is built for low latency.", "document_id": "doc_0",
     "quote": "optimized for low latency and high throughput"},               # wrong doc
    {"claim": "Flash has a 1 million token context.", "document_id": "doc_1",
     "quote": "context window of up to 1 million tokens for Flash"},          # frankenquote
    {"claim": "Gemini 2.5 Pro runs fully offline.", "document_id": "doc_0",
     "quote": "runs entirely on-device with no network"},                     # fabricated
]

for c in CITATIONS:
    status = gate(c["quote"], c["document_id"], DOCS)
    flag = "OK  " if status == "found" else "FLAG"
    print(f"{flag} [{status:>13}]  {c['claim']}")

The gate settles one question only: does this quote appear, verbatim, in the
document it is attributed to? `misattributed` and `not_found` can be flagged or
dropped right here, deterministically, for zero tokens. A `found` citation is not
yet *correct* — a real, correctly-attributed quote can still fail to support the
claim it is attached to. That judgment is what the optional judge below is for.

## Generate the citations with Gemini structured output

With an API key, ask Gemini to answer **and** return structured citations, then run
the same gate over them. Uses the `google-genai` SDK with a `response_schema`; needs
`GEMINI_API_KEY` (not run in CI).

In [ ]:
%pip install -U -q "google-genai>=2.9.0" pydantic
import os

MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash", "gemini-2.5-pro"]

if not os.getenv("GEMINI_API_KEY"):
    print("Set GEMINI_API_KEY to run the live example.")
else:
    from google import genai
    from pydantic import BaseModel

    class Citation(BaseModel):
        claim: str
        document_id: str
        quote: str

    class CitedAnswer(BaseModel):
        answer: str
        citations: list[Citation]

    client = genai.Client()
    library = "\n".join(f"[{doc_id}] {text}" for doc_id, text in DOCS.items())
    prompt = (
        "Answer using only the library. For each claim, cite the document_id and a short quote "
        f"copied verbatim from that document.\n\nLibrary:\n{library}\n\n"
        "Question: What context window does Gemini 2.5 Pro have, and what is Flash optimized for?"
    )
    interaction = client.interactions.create(
        model=MODEL_ID,
        input=prompt,
        config={"response_mime_type": "application/json", "response_schema": CitedAnswer},
    )
    cited = CitedAnswer.model_validate_json(interaction.steps[-1].content[0].text)
    print(cited.answer, "\n")
    for c in cited.citations:
        status = gate(c.quote, c.document_id, DOCS)
        flag = "OK  " if status == "found" else "FLAG"
        print(f"{flag} [{status:>13}]  {c.quote!r} -> {c.document_id}")

## Optional: a burden-of-proof judge for the ambiguous case

The gate settles whether a quote *exists*. Whether a real, correctly-attributed quote
actually **supports** its claim is a judgment call best given to a model — but with
the burden of proof on the citation: default to *unsupported*, outside knowledge is
inadmissible, and an unparseable verdict fails closed. Because only `found` quotes
reach it, fabricated and misattributed citations cost zero judge calls.

A ready-made, model-agnostic version of this judge is in
[`verbatim-citation-gate`](https://github.com/Palo-Alto-AI-Research-Lab/verbatim-citation-gate);
plug your `client.models.generate_content` call into its `llm_call` hook.